# Calcite–dolomite column with xGEM2MT

A 1-D column of 51 boxes filled with calcite is flushed with a MgCl$_2$ solution. Calcite dissolves and dolomite forms, and the dolomitization front moves along the column.

The mass-transport scheme is **'F'**: a chain of flow-through reactors connected by 1st order fluxes. At each time step, part of the fluid moves from each box into the next one, then all boxes are equilibrated with GEMS3K.

The input files (`CalcColumn/`) were exported from the GEMS GUI project `CalDolCol2` (GEM2MT task `CalcColumn`, test `Test2:F`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from xgem2mt import MassTransport

## Collect results during the run

The monitoring function is called after each time step with the array of chemical nodes. Here we keep calcite, dolomite and pH in all boxes every 100 steps.

In [ ]:
results = {}

def monitor(engine, step, time):
    if step % 100 == 0:
        nodes = range(engine.nNodes())
        results[time] = {
            "Calcite":  [engine.phaseAmount(n, "Calcite") for n in nodes],
            "Dolomite": [engine.phaseAmount(n, "Dolomite-dis") for n in nodes],
            "pH":       [engine.pH(n) for n in nodes],
        }

## Set up the task

In [ ]:
nnodes = 51

# (1) Create the MassTransport simulator in 'F' mode
mt_task = MassTransport("F", nnodes)
mt_task.setName("Calcite-dolomite column (flow-through reactors)")

# (2) Time iterator (start, end, step), one mobile phase group, one flux per box
mt_task.setTau(0.0, 1200.0, 1.0)
mt_task.setNumberPhaseGroups(1)
mt_task.setNumberMGPfluxes(nnodes)
mt_task.setSIA("-")            # automatic initial approximation
mt_task.setTypeFluxPhase("1")  # aqueous phase is mobile
mt_task.setOutput(False)
mt_task.setMonitoring(monitor)

# (3) Read the GEMS3K system files exported from GEMS
mt_task.initNodes("CalcColumn/CalcColumn-dat.lst")

`print(mt_task)` (or `mt_task.summary()`) shows the setup: the column, the time step that the program uses, which nodes are fluid and column, their properties and phases, and which function changes each value.

In [ ]:
print(mt_task)

Define the mobile phase group (the aqueous phase) and the fluxes from box `i` to box `i+1`; the last box flows out of the column.

In [ ]:
mt_task.setPhaseGroupID(0, "phg1")
for k in range(mt_task.nPhases):
    mt_task.setPhaseGroupQuantities(0, k, 1.0 if k == 0 else 0.0)
    mt_task.setUnitsPhaseQuantities(k, "M")

for i in range(mt_task.nMGPfluxes):
    mt_task.setFluxSourceReceive(i, i, i + 1 if i < nnodes - 1 else -1)
    mt_task.setFluxDefinition(i, 1.0, 0.5, 0, 0)  # 1st order, rate 0.5
    mt_task.setFluxMGPid(i, "phg1")
    mt_task.setFluxID(i, "qj")

## Run the simulation

In [ ]:
mt_task.runMassTransport()
times = [t for t in sorted(results) if t % 300 == 0]
boxes = np.arange(nnodes)

## Plot the results

Calcite in the column:

In [ ]:
for t in times:
    plt.plot(boxes, results[t]["Calcite"], label=f"step {t:.0f}")
plt.xlabel("Box"); plt.ylabel("Calcite, mol"); plt.legend(); plt.show()

Dolomite in the column:

In [ ]:
for t in times:
    plt.plot(boxes, results[t]["Dolomite"], label=f"step {t:.0f}")
plt.xlabel("Box"); plt.ylabel("Dolomite, mol"); plt.legend(); plt.show()

pH of the pore fluid:

In [ ]:
for t in times:
    plt.plot(boxes, results[t]["pH"], label=f"step {t:.0f}")
plt.xlabel("Box"); plt.ylabel("pH"); plt.legend(); plt.show()